
# AI Leadership Insight & Decision Agent

An AI-powered assistant that ingests company documents (annual reports, quarterly reports, strategy notes, operational updates) and answers leadership questions in natural language — grounded entirely in the provided documents.

---

## Architecture Overview

```
Company Documents (PDF / TXT / DOCX)
        │
        ▼
  [Document Loader]
        │
        ▼
  [Text Chunker]   ← RecursiveCharacterTextSplitter
        │
        ▼
  [Embeddings]     ← sentence-transformers (all-MiniLM-L6-v2)
        │
        ▼
  [Vector Store]   ← FAISS in-memory index
        │
   Question ──────► [Retriever] → Top-K relevant chunks
                         │
                         ▼
                  [LLM: Claude claude-sonnet-4-20250514]
                         │
                         ▼
               Grounded Leadership Answer
```

**Stack:** Python · LangChain · FAISS · sentence-transformers · Anthropic Claude API

## Step 1 — Install Dependencies

In [ ]:
# Install all required packages
!pip install -q \
    langchain \
    langchain-community \
    langchain-text-splitters \
    faiss-cpu \
    sentence-transformers \
    pypdf \
    python-docx \
    tiktoken \
    gradio

print(" All packages installed.")

 All packages installed.


## Step 2 — Configuration & API Key

In [ ]:
import os
from getpass import getpass

# ── Set your OpenAI API key ──────────────────────────────────────────────
# Option A: enter interactively
OPENAI_API_KEY = getpass(" Enter your OpenAI API key: ")
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

# ── Agent configuration ─────────────────────────────────────────────────────
CONFIG = {
    "model": "gpt-4o-mini",          # Open AI model to use
    "max_tokens": 1024,                             # Max tokens in LLM response
    "chunk_size": 800,                              # Characters per text chunk
    "chunk_overlap": 150,                           # Overlap between chunks
    "top_k": 5,                                     # Number of chunks to retrieve
    "embedding_model": "all-MiniLM-L6-v2",          # Sentence-transformer model
}

print(" Configuration loaded.")
print(f"   Model      : {CONFIG['model']}")
print(f"   Top-K      : {CONFIG['top_k']} chunks retrieved per query")
print(f"   Chunk size : {CONFIG['chunk_size']} chars  (overlap: {CONFIG['chunk_overlap']})")

 Enter your OpenAI API key: ··········
 Configuration loaded.
   Model      : gpt-4o-mini
   Top-K      : 5 chunks retrieved per query
   Chunk size : 800 chars  (overlap: 150)


## Step 3 — Document Ingestion

Supports **.pdf**, **.txt**, and **.docx** files.  
Upload your company documents using the helper below, **or** use the built-in sample documents to test immediately.

Here we are using Adobe Annual Report openly available for reference
https://www.google.com/url?sa=t&source=web&rct=j&opi=89978449&url=https://www.adobe.com/cc-shared/assets/investor-relations/pdfs/adbe-2024-annual-report.pdf&ved=2ahUKEwjX5szU9qaTAxXSUWwGHYSzL98QFnoECB8QAQ&usg=AOvVaw3b3at45OPxDa6JzsF7Rfre

In [ ]:
import io
import textwrap
from pathlib import Path

import pypdf
import docx as python_docx


# ── Document loaders ────────────────────────────────────────────────────────

def load_pdf(path: str) -> str:
    """Extract text from a PDF file."""
    reader = pypdf.PdfReader(path)
    return "\n".join(page.extract_text() or "" for page in reader.pages)


def load_docx(path: str) -> str:
    """Extract text from a .docx file."""
    doc = python_docx.Document(path)
    return "\n".join(p.text for p in doc.paragraphs)


def load_txt(path: str) -> str:
    """Read a plain-text file."""
    return Path(path).read_text(encoding="utf-8", errors="ignore")


def load_document(path: str) -> dict:
    """Auto-detect file type and return {source, content}."""
    ext = Path(path).suffix.lower()
    loaders = {".pdf": load_pdf, ".docx": load_docx, ".txt": load_txt}
    if ext not in loaders:
        raise ValueError(f"Unsupported file type: {ext}")
    content = loaders[ext](path)
    return {"source": Path(path).name, "content": content}


def load_documents_from_folder(folder: str) -> list[dict]:
    """Load all supported documents from a folder."""
    docs = []
    for ext in ("*.pdf", "*.docx", "*.txt"):
        for fp in Path(folder).glob(ext):
            doc = load_document(str(fp))
            docs.append(doc)
            print(f"  📄 Loaded: {doc['source']}  ({len(doc['content'])} chars)")
    return docs


print(" Document loaders defined.")

 Document loaders defined.


In [ ]:
# ── Option A: Upload your own documents ─────────────────────────────────────
from google.colab import files
uploaded = files.upload()          # select one or more files
UPLOAD_DIR = "/content/company_docs"
os.makedirs(UPLOAD_DIR, exist_ok=True)
for name, data in uploaded.items():
    with open(f"{UPLOAD_DIR}/{name}", "wb") as f:
        f.write(data)
raw_docs = load_documents_from_folder(UPLOAD_DIR)

for d in raw_docs:
    print(f"   {d['source']}  ({len(d['content'])} chars)")

Saving adbe-2024-annual-report.pdf to adbe-2024-annual-report.pdf
  📄 Loaded: adbe-2024-annual-report.pdf  (423644 chars)
   📄 adbe-2024-annual-report.pdf  (423644 chars)


## Step 4 — Chunking & Embedding

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np


# ── 4a. Chunking ─────────────────────────────────────────────────────────────

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CONFIG["chunk_size"],
    chunk_overlap=CONFIG["chunk_overlap"],
    separators=["\n\n", "\n", ". ", " ", ""],
)

chunks = []  # list of {source, chunk_id, text}
for doc in raw_docs:
    splits = splitter.split_text(doc["content"])
    for i, text in enumerate(splits):
        chunks.append({"source": doc["source"], "chunk_id": i, "text": text})

print(f" Created {len(chunks)} chunks from {len(raw_docs)} documents.")

# Preview first chunk
print("\n── Sample chunk ──")
print(f"Source : {chunks[0]['source']}")
print(f"Text   : {chunks[0]['text'][:300]}...")

 Created 669 chunks from 1 documents.

── Sample chunk ──
Source : adbe-2024-annual-report.pdf
Text   : UNITED STATES
SECURITIES AND EXCHANGE COMMISSION
Washington, D.C. 20549
_____________________________
 FORM 10-K 
(Mark One)
☒ ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934
For the fiscal year ended November 29, 2024
 or
☐ TRANSITION REPORT PURSUANT TO SECTION ...


In [ ]:
# ── 4b. Embedding ─────────────────────────────────────────────────────────────

print(f"Loading embedding model: {CONFIG['embedding_model']} ...")
embedder = SentenceTransformer(CONFIG["embedding_model"])

texts = [c["text"] for c in chunks]
print(f"Encoding {len(texts)} chunks...")
embeddings = embedder.encode(texts, show_progress_bar=True, convert_to_numpy=True)

print(f"\n Embeddings shape: {embeddings.shape}")


# ── 4c. Build FAISS index ─────────────────────────────────────────────────────

dim = embeddings.shape[1]
index = faiss.IndexFlatL2(dim)           # L2 distance (equivalent to cosine on normalised vecs)
faiss.normalize_L2(embeddings)           # Normalise → cosine similarity
index.add(embeddings)

print(f" FAISS index built with {index.ntotal} vectors (dim={dim}).")

Loading embedding model: all-MiniLM-L6-v2 ...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Encoding 669 chunks...


Batches:   0%|          | 0/21 [00:00<?, ?it/s]


 Embeddings shape: (669, 384)
 FAISS index built with 669 vectors (dim=384).


## Step 5 — Retrieval & Generation

In [ ]:
import openai

client = openai.OpenAI(api_key=os.environ["OPENAI_API_KEY"])


# ── Retriever ─────────────────────────────────────────────────────────────────

def retrieve(query: str, top_k: int = CONFIG["top_k"]) -> list[dict]:
    """Embed the query and return the top-K most relevant chunks."""
    q_emb = embedder.encode([query], convert_to_numpy=True)
    faiss.normalize_L2(q_emb)
    distances, indices = index.search(q_emb, top_k)
    results = []
    for dist, idx in zip(distances[0], indices[0]):
        if idx == -1:
            continue
        chunk = chunks[idx].copy()
        chunk["similarity"] = float(1 - dist / 2)  # convert L2 distance → cosine similarity
        results.append(chunk)
    return results


# ── Prompt builder ────────────────────────────────────────────────────────────

SYSTEM_PROMPT = """You are a senior business analyst serving company leadership.
You answer questions ONLY using the context excerpts provided — do not add outside knowledge.
If the context does not contain enough information to answer, say so clearly.

Guidelines:
- Be concise but complete. Use bullet points for lists of items.
- Cite the source document for each key fact (e.g. [Q3_2024_Earnings.txt]).
- If numbers are mentioned, be precise — do not round unless necessary.
- Highlight risks, red flags, or underperformance prominently.
"""


def build_prompt(question: str, context_chunks: list[dict]) -> str:
    context_text = "\n\n".join(
        f"[Source: {c['source']}]\n{c['text']}"
        for c in context_chunks
    )
    return f"""Use the following excerpts from company documents to answer the leadership question.

=== CONTEXT EXCERPTS ===
{context_text}

=== LEADERSHIP QUESTION ===
{question}

Provide a clear, evidence-based answer citing the relevant documents."""


# ── Main agent function ───────────────────────────────────────────────────────

def ask_agent(question: str, verbose: bool = False) -> str:
    """
    Ask the leadership insight agent a natural-language question.
    Returns a grounded answer citing source documents.
    """
    # 1. Retrieve relevant chunks
    context = retrieve(question)

    if verbose:
        print(f"\n Retrieved {len(context)} chunks:")
        for c in context:
            print(f"   [{c['similarity']:.2f}] {c['source']} | {c['text'][:80]}...")
        print()

    # 2. Build prompt
    user_prompt = build_prompt(question, context)

    # 3. Call OpenAI
    response = client.chat.completions.create(
        model=CONFIG["model"],
        max_tokens=CONFIG["max_tokens"],
        messages=[{"role": "system", "content": SYSTEM_PROMPT},
{"role": "user", "content": user_prompt}],
    )

    return response.choices[0].message.content


print(" Agent is ready. Use ask_agent('your question') to query.")

 Agent is ready. Use ask_agent('your question') to query.


## Step 6 — Ask Leadership Questions

Run each cell to see how the agent answers real leadership questions.

In [ ]:
# ── Question 1: Revenue trend ─────────────────────────────────────────────────

q1 = "What is our current revenue trend?"
print(f" {q1}\n{'─'*60}")
print(ask_agent(q1, verbose=True))

 What is our current revenue trend?
────────────────────────────────────────────────────────────

 Retrieved 5 chunks:
   [0.62] adbe-2024-annual-report.pdf | discussion of the possible impact of these macroeconomic issues on our business....
   [0.60] adbe-2024-annual-report.pdf | $17.22 billion or approximately $117 million lower than the ARR reported above.
...
   [0.58] adbe-2024-annual-report.pdf | market, and our recent financial results reflect the success of these investment...
   [0.56] adbe-2024-annual-report.pdf | in fiscal 2023.
• Cost of revenue of $2.36 billion during fiscal 2024 remained r...
   [0.54] adbe-2024-annual-report.pdf | We generally categorize revenue by geographic area based on where the customer m...

The current revenue trend shows positive growth across multiple segments for fiscal 2024:

- **Total Digital Media Annual Recurring Revenue (ARR)**: Approximately $17.33 billion, up $2.00 billion (13%) from $15.33 billion in fiscal 2023. [adbe-2024-annual-repo

In [ ]:
# ── Question 2: Underperforming departments ───────────────────────────────────

q2 = "Which departments are underperforming and why?"
print(f" {q2}\n{'─'*60}")
print(ask_agent(q2, verbose=True))

 Which departments are underperforming and why?
────────────────────────────────────────────────────────────

 Retrieved 5 chunks:
   [0.33] adbe-2024-annual-report.pdf | Percentage of total revenue   .....................................................
   [0.32] adbe-2024-annual-report.pdf | diversity and inclusion. As of November 29, 2024, we employed 30,709 people, of ...
   [0.31] adbe-2024-annual-report.pdf | continue to invest in digital transformation. 
SEGMENTS
Our business is organize...
   [0.31] adbe-2024-annual-report.pdf | Workfront
All other trademarks are the property of their respective owners.
100
...
   [0.31] adbe-2024-annual-report.pdf | Shantanu Narayen Chair of the Board of Directors and 
Chief Executive Officer
(P...

From the excerpts provided, there are no specific indications of underperformance in any departments. However, here are some observations:

- **Sales and Marketing**: 
  - While sales and marketing expenses increased from $4,968 million to $5,764 m

In [ ]:
# ── Question 3: Key risks ─────────────────────────────────────────────────────

q3 = "What were the key risks highlighted in the last quarter?"
print(f" {q3}\n{'─'*60}")
print(ask_agent(q3, verbose=True))

 What were the key risks highlighted in the last quarter?
────────────────────────────────────────────────────────────

 Retrieved 5 chunks:
   [0.45] adbe-2024-annual-report.pdf | for,” “looks to,” “continues” and similar expressions, as well as statements reg...
   [0.45] adbe-2024-annual-report.pdf | 22
ITEM 1A.  RISK FACTORS
As previously discussed, our actual results could diff...
   [0.42] adbe-2024-annual-report.pdf | 33
The occurrence of an epidemic or a pandemic, such as the COVID-19 pandemic, h...
   [0.39] adbe-2024-annual-report.pdf | future and affect the terms of any such financing.
General Risk Factors
Catastro...
   [0.37] adbe-2024-annual-report.pdf | United States (“GAAP”) and pursuant to the rules and regulations of the SEC, we ...

The key risks highlighted in the last quarter include:

- **Potential Impact of Epidemics or Pandemics**:
  - The occurrence of epidemics or pandemics, similar to COVID-19, has adversely affected operating results and may continue to do s

In [ ]:
# ── Question 4: Strategic priorities ─────────────────────────────────────────

q4 = "What are our top strategic priorities for 2025?"
print(f" {q4}\n{'─'*60}")
print(ask_agent(q4))

 What are our top strategic priorities for 2025?
────────────────────────────────────────────────────────────
The top strategic priorities for Adobe in 2025 are focused on the following key areas:

- **Investing in Digital Transformation**: Continuing to enhance and expand digital solutions is essential, as evidenced by the emphasis on ongoing investment in this area [adbe-2024-annual-report.pdf].
  
- **Focus on Key Segments**: Organizational alignment around three major reportable segments:
  - Digital Media
  - Digital Experience
  - Publishing and Advertising  
  These segments will help Adobe leverage strategic growth opportunities while managing legacy products [adbe-2024-annual-report.pdf].

- **Financial Stability and Predictability**: Maintaining the subscription-based revenue model to ensure consistent financial performance, while being attuned to macroeconomic factors that might impact operations and investments [adbe-2024-annual-report.pdf].

In summary, Adobe's strategic p

In [ ]:
# ── Question 5: People & talent ───────────────────────────────────────────────

q5 = "What are our current people and talent challenges?"
print(f" {q5}\n{'─'*60}")
print(ask_agent(q5))

 What are our current people and talent challenges?
────────────────────────────────────────────────────────────
Our current people and talent challenges include:

- **Intense Competition for Talent**: 
  - There is substantial and ongoing competition for talent, particularly in the areas of AI and cybersecurity, which presents difficulties in recruiting and retaining skilled personnel ([adbe-2024-annual-report.pdf]).
  - Increased competition may lead to heightened compensation costs, which might not be offset by gains in innovation, productivity, or sales ([adbe-2024-annual-report.pdf]).

- **Hybrid Work Environment Risks**:
  - The hybrid work setup can lead to operational, security, and workplace culture challenges, potentially undermining our ability to execute business objectives and retain employees ([adbe-2024-annual-report.pdf]).

- **Succession Planning and Management Continuity**:
  - Our future success is highly dependent on the service and performance of senior management 

In [ ]:
# ── Question 6: Cloud Services performance ────────────────────────────────────

q6 = "How is the Cloud Services division performing and what is the growth outlook?"
print(f" {q6}\n{'─'*60}")
print(ask_agent(q6))

 How is the Cloud Services division performing and what is the growth outlook?
────────────────────────────────────────────────────────────
The Cloud Services division, particularly represented by the Digital Media segment, is experiencing positive performance with significant revenue growth, indicating a strong growth outlook.

### Current Performance:
- **Total Digital Media ARR**: Approximately $17.33 billion as of November 29, 2024, increasing by $2.00 billion or 13% from $15.33 billion as of December 1, 2023 [adbe-2024-annual-report.pdf].
- **Creative Revenue**: $12.68 billion in fiscal 2024, a year-over-year increase of $1.17 billion or 10% from $11.52 billion in fiscal 2023 [adbe-2024-annual-report.pdf].
- **Document Cloud Revenue**: $3.18 billion, up $483 million or 18% from $2.70 billion in fiscal 2023 [adbe-2024-annual-report.pdf].
- **Digital Experience Revenue**: $5.37 billion, an increase of $473 million or 10% from $4.89 billion in fiscal 2023 [adbe-2024-annual-report.pdf

## Step 7 — Interactive Chat Interface

Launch a Gradio UI for live Q&A with the leadership agent.

In [ ]:
import gradio as gr

EXAMPLE_QUESTIONS = [
    "What is our current revenue trend?",
    "Which departments are underperforming?",
    "What were the key risks highlighted in the last quarter?",
    "What are our top strategic priorities for 2025?",
    "How is the Cloud Services division performing?",
    "What is our customer churn rate and what are we doing about it?",
    "What major deals or wins did we close recently?",
    "What is the status of the Hardware division?",
]


def chat(question, history):
    """Gradio chat handler."""
    if not question.strip():
        return history, history
    answer = ask_agent(question)
    history.append((question, answer))
    return history, history


with gr.Blocks(
    title="AI Leadership Insight Agent",
    theme=gr.themes.Soft(),
    css=".gradio-container { max-width: 900px; margin: auto }",
) as demo:

    gr.Markdown("""#  AI Leadership Insight Agent
    Ask any question about company performance, risks, or strategy.
    Answers are grounded in uploaded company documents.
    """)

    chatbot = gr.Chatbot(height=450, label="Conversation")
    state = gr.State([])

    with gr.Row():
        question_box = gr.Textbox(
            placeholder="Ask a leadership question...",
            label="Your question",
            scale=4,
        )
        submit_btn = gr.Button("Ask", variant="primary", scale=1)

    gr.Examples(
        examples=EXAMPLE_QUESTIONS,
        inputs=question_box,
        label="Example questions",
    )

    clear_btn = gr.Button("Clear conversation", variant="secondary")

    submit_btn.click(
        chat,
        inputs=[question_box, state],
        outputs=[chatbot, state],
    ).then(lambda: "", outputs=question_box)

    question_box.submit(
        chat,
        inputs=[question_box, state],
        outputs=[chatbot, state],
    ).then(lambda: "", outputs=question_box)

    clear_btn.click(lambda: ([], []), outputs=[chatbot, state])

demo.launch(share=True)  # share=True gives a public URL in Colab

/tmp/ipykernel_275/3303937646.py:24: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(
/tmp/ipykernel_275/3303937646.py:24: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(
/tmp/ipykernel_275/3303937646.py:35: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(height=450, label="Conversation")
/tmp/ipykernel_275/3303937646.py:35: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitl

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c75b81c8788bb0ed17.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Step 8 — Evaluation: Retrieval Quality Check

In [ ]:
# Quick evaluation: show which chunks are retrieved for each sample question
# and their similarity scores. Useful for debugging retrieval quality.

EVAL_QUESTIONS = [
    "What is our current revenue trend?",
    "Which departments are underperforming?",
    "What were the key risks highlighted in the last quarter?",
    "What are our strategic priorities for 2025?",
]

print("=" * 70)
print("RETRIEVAL QUALITY EVALUATION")
print("=" * 70)

for q in EVAL_QUESTIONS:
    print(f"\n {q}")
    results = retrieve(q, top_k=3)
    for r in results:
        bar = "█" * int(r["similarity"] * 20)
        print(f"   {r['similarity']:.3f} {bar:<20} | {r['source']}")
        print(f"            {r['text'][:100].strip()}...")
    print()

RETRIEVAL QUALITY EVALUATION

 What is our current revenue trend?
   0.617 ████████████         | adbe-2024-annual-report.pdf
            discussion of the possible impact of these macroeconomic issues on our business.
Financial Performan...
   0.603 ████████████         | adbe-2024-annual-report.pdf
            $17.22 billion or approximately $117 million lower than the ARR reported above.
Our success in drivi...
   0.576 ███████████          | adbe-2024-annual-report.pdf
            market, and our recent financial results reflect the success of these investments and our experience...


 Which departments are underperforming?
   0.340 ██████               | adbe-2024-annual-report.pdf
            Percentage of total revenue   ...............................................................  18 %...
   0.330 ██████               | adbe-2024-annual-report.pdf
            diversity and inclusion. As of November 29, 2024, we employed 30,709 people, of which 50% were in th...
   0.316 ████

---

## Architecture Summary

| Component | Choice | Rationale |
|---|---|---|
| **LLM** | gpt-4o-mini | Strong instruction-following, accurate citations |
| **Embeddings** | all-MiniLM-L6-v2 | Fast, free, runs on CPU, 384-dim vectors |
| **Vector store** | FAISS (IndexFlatL2 + L2 normalisation) | In-memory, no server needed, exact cosine search |
| **Chunking** | RecursiveCharacterTextSplitter (800 chars, 150 overlap) | Preserves sentence boundaries; overlap reduces context loss |
| **File formats** | PDF · DOCX · TXT | Covers all common company document types |
| **UI** | Gradio chat | Zero-config interactive interface |

## Extending This Agent

- **Larger documents / more files** → swap FAISS for [ChromaDB](https://www.trychroma.com/) or [Pinecone](https://www.pinecone.io/) for persistent storage  
- **Better embeddings** → use `text-embedding-3-large` (OpenAI) or `voyage-2` (Anthropic) for higher accuracy  
- **Hybrid search** → combine dense embeddings with BM25 keyword search (via `rank_bm25`)  
- **Multi-turn conversation** → maintain message history and pass previous Q&A into Claude's context  
- **Source highlighting** → return exact page numbers / paragraph offsets from PDF/DOCX